# Visualize Intermediate Domains on FLIR: RGB | IR | IR_GAN | GAN-Low + RandomMono-High | SwitchMix FDA

Notebook độc lập (**không cần cài thêm package, chạy thuần PyTorch/torchvision**) để đối chiếu trực quan trên **cùng 1 hàng**:
1. **RGB** (ảnh gốc)
2. **IR Aligned** (ảnh nhiệt thật để đối chiếu)
3. **IR_GAN gốc** (đầu ra CycleGAN `latest_net_G_A.pth`)
4. **Hướng 2 — GAN-Low + RandomMono-High**: lấy phân bố sáng/nhiệt tần số thấp từ `IR_GAN` + ghép cạnh sắc nét tần số cao từ `RandomMono` (khắc phục lỗi nhòe vật thể nhỏ như người & xe đạp)
5. **SwitchMix FDA (`β = 0.01`)**: ghép biên độ tần số thấp của 1 ảnh IR thật khác vào `RandomMono` (đúng Eq. 5 trong paper SwitchMix)

Ở **Cell cuối**, notebook còn tự động **zoom sát vào vật thể nhỏ (`person` / `bicycle`)** và so sánh các mức `sigma` (`1.5`, `3.0`, `5.0`) để bạn chọn tham số đẹp nhất.

**Cách dùng trên Kaggle**: gắn dataset FLIR + dataset chứa `latest_net_G_A.pth`, bật **GPU** (không cần Internet), kiểm tra 2 đường dẫn ở **Cell 1 (`CONFIG`)**, rồi bấm **Run All**.


In [ ]:
# Cell 1: Toàn bộ cấu hình (chỉ cần chỉnh ở đây)
CONFIG = {
    "data_root": "/kaggle/input/datasets/tungduong123qwe/flir-dataset/align",
    "gan_weights": "/kaggle/input/datasets/bdv212/generator-rgb-ir/latest_net_G_A.pth",
    "split": "train",             # "train" hoặc "validation"
    "num_images": 5,              # Số hàng hiển thị (mỗi hàng 5 cột)
    "start_index": 0,             # Bỏ qua N ảnh hợp lệ đầu tiên
    "only_with_boxes": True,      # True = chỉ lấy ảnh có ít nhất 1 GT box
    "prioritize_small": True,     # True = ưu tiên ảnh có person hoặc bicycle để soi vật thể nhỏ
    "show_gt_boxes": False,       # True = vẽ khung GT lên cột 1 (RGB) để dễ nhận diện vật thể nhỏ
    "device": "cuda",             # Tự động fallback sang "cpu" nếu không bật GPU
    "alpha": 0.0,                 # 0.0 = 100% miền trung gian; 0.25 / 0.50 / 0.75 = mức pha trong train
    # --- Tham số Hướng 2: Low(IR_GAN) + lambda * High(RandomMono) ---
    "fusion_sigma": 3.0,          # Độ rộng Gaussian tách Low/High (thử 1.5, 3.0, 5.0)
    "fusion_high_weight": 1.0,    # Hệ số lambda nhân với cạnh sắc nét High(RandomMono) (thử 0.7, 1.0, 1.3)
    # --- Tham số đối chiếu SwitchMix FDA (Eq. 5) ---
    "fda_beta": 0.01,             # Đúng beta=0.01 của paper SwitchMix
    "fda_phi": 1.0,               # Tỷ lệ thay biên độ tần số thấp (0..1)
    "min_box_area": 16.0,
    "seed": 42,
    "save_path": "/kaggle/working/domain_comparison_5cols.png",
    "zoom_save_path": "/kaggle/working/small_object_sigma_sweep.png",
}


In [ ]:
# Cell 2: CycleGAN loader (khớp 100% repo), đọc FLIR aligned, Hướng 2 (GAN-Low + RandomMono-High) & SwitchMix FDA
import math
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
import torchvision.transforms.functional as TF

FLIR_CLASSES = {"person", "car", "bicycle"}


class _ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, images):
        return images + self.conv_block(images)


class CycleGANGenerator(nn.Module):
    """Khớp 100% models/cyclegan_generator.py (ResNet-9, RGB->IR 1ch)."""

    def __init__(self, ngf: int = 64):
        super().__init__()
        layers = [nn.ReflectionPad2d(3), nn.Conv2d(3, ngf, 7),
                  nn.InstanceNorm2d(ngf), nn.ReLU(inplace=True)]
        channels = ngf
        for _ in range(2):
            layers.extend([nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1),
                           nn.InstanceNorm2d(channels * 2), nn.ReLU(inplace=True)])
            channels *= 2
        layers.extend(_ResidualBlock(channels) for _ in range(9))
        for _ in range(2):
            layers.extend([nn.ConvTranspose2d(channels, channels // 2, 3, stride=2, padding=1, output_padding=1),
                           nn.InstanceNorm2d(channels // 2), nn.ReLU(inplace=True)])
            channels //= 2
        layers.extend([nn.ReflectionPad2d(3), nn.Conv2d(ngf, 1, 7), nn.Tanh()])
        self.model = nn.Sequential(*layers)

    def forward(self, images):
        return self.model(images)


class FrozenGANTranslator(nn.Module):
    """Khớp 100% training/gan_bridge.py."""

    def __init__(self, generator: nn.Module):
        super().__init__()
        self.generator = generator
        self.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        height, width = images.shape[-2:]
        pad_h = max(8, (height + 3) // 4 * 4) - height
        pad_w = max(8, (width + 3) // 4 * 4) - width
        outputs = []
        for image in images.split(1):
            gan_input = F.pad(image * 2 - 1, (0, pad_w, 0, pad_h), mode="replicate")
            output = self.generator(gan_input)
            thermal = ((output[:, :, :height, :width] + 1) * 0.5).clamp(0, 1)
            outputs.append(thermal.repeat(1, 3, 1, 1))
        return torch.cat(outputs, dim=0)


def load_gan_translator(checkpoint: str, device: torch.device) -> FrozenGANTranslator:
    path = Path(checkpoint)
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy GAN checkpoint: {path}")
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if all(key.startswith("module.") for key in state):
        state = {key.removeprefix("module."): value for key, value in state.items()}
    generator = CycleGANGenerator()
    generator.load_state_dict(state, strict=True)
    return FrozenGANTranslator(generator).to(device)


def resolve_split_file(root: Path, split: str) -> Path:
    filename = f"align_{split}.txt"
    for candidate in (root / filename, root / "ImageSets" / "Main" / filename):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Không tìm thấy {filename} trong {root}")


def parse_flir_objects(xml_path: Path, min_area: float = 16.0):
    if not xml_path.is_file():
        return []
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return []
    objects = []
    for obj in root.findall("object"):
        name = obj.findtext("name", default="").strip()
        if name not in FLIR_CLASSES:
            continue
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        try:
            x1 = float(bnd.findtext("xmin"))
            y1 = float(bnd.findtext("ymin"))
            x2 = float(bnd.findtext("xmax"))
            y2 = float(bnd.findtext("ymax"))
        except (TypeError, ValueError):
            continue
        area = (x2 - x1) * (y2 - y1)
        if x2 > x1 and y2 > y1 and area >= min_area:
            objects.append({"label": name, "box": [x1, y1, x2, y2], "area": area})
    return objects


def load_flir_samples(cfg: dict):
    root = Path(cfg["data_root"])
    split_file = resolve_split_file(root, cfg["split"])
    stems = [line.strip() for line in split_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    samples = []
    skipped = 0
    for stem in stems:
        rgb_path = root / "JPEGImages" / (stem.replace("_PreviewData", "") + "_RGB.jpg")
        ir_path = root / "JPEGImages" / f"{stem}.jpeg"
        xml_path = root / "Annotations" / f"{stem}.xml"
        if not (rgb_path.is_file() and ir_path.is_file()):
            continue
        objects = parse_flir_objects(xml_path, min_area=cfg["min_box_area"])
        if cfg["only_with_boxes"] and not objects:
            continue
        if cfg["prioritize_small"] and not any(o["label"] in {"person", "bicycle"} for o in objects):
            continue
        if skipped < cfg["start_index"]:
            skipped += 1
            continue
        rgb_np = np.asarray(Image.open(rgb_path).convert("RGB"), dtype=np.float32) / 255.0
        ir_np = np.asarray(Image.open(ir_path).convert("L"), dtype=np.float32) / 255.0
        samples.append({
            "stem": stem,
            "rgb_t": torch.from_numpy(rgb_np).permute(2, 0, 1).float(),
            "ir_t": torch.from_numpy(ir_np).unsqueeze(0).repeat(3, 1, 1).float(),
            "objects": objects,
        })
        if len(samples) >= cfg["num_images"]:
            break
    if not samples:
        raise RuntimeError(f"Không tìm thấy mẫu hợp lệ nào trong {root} (split={cfg['split']}).")
    return samples


def _gaussian_blur(tensor_chw: torch.Tensor, sigma: float) -> torch.Tensor:
    if sigma <= 0:
        return tensor_chw.clone()
    ksize = max(3, int(2 * math.ceil(3.0 * sigma) + 1))
    return TF.gaussian_blur(tensor_chw.unsqueeze(0), kernel_size=[ksize, ksize], sigma=[sigma, sigma]).squeeze(0)


def make_randommono(rgb_t: torch.Tensor):
    weights = torch.rand(3, 1, 1, device=rgb_t.device, dtype=rgb_t.dtype)
    weights = weights / weights.sum().clamp_min(1e-12)
    mono = (rgb_t * weights).sum(dim=0, keepdim=True).expand_as(rgb_t)
    w_tuple = tuple(round(float(w), 2) for w in weights.view(-1))
    return mono, w_tuple


def apply_gan_low_randommono_high(
    rgb_t: torch.Tensor,
    gan_t: torch.Tensor,
    mono_t: torch.Tensor,
    *,
    alpha: float = 0.0,
    sigma: float = 3.0,
    high_weight: float = 1.0,
) -> torch.Tensor:
    """Hướng 2: Low-pass (thermal tone/contrast) từ GAN + High-pass (crisp edges) từ RandomMono."""
    low_gan = _gaussian_blur(gan_t, sigma)
    high_mono = mono_t - _gaussian_blur(mono_t, sigma)
    endpoint = (low_gan + high_weight * high_mono).clamp(0.0, 1.0)
    return (alpha * rgb_t + (1.0 - alpha) * endpoint).clamp(0.0, 1.0)


def apply_switchmix_fda(
    source_t: torch.Tensor,
    target_ir_t: torch.Tensor,
    *,
    beta: float = 0.01,
    phi: float = 1.0,
    alpha: float = 0.0,
) -> torch.Tensor:
    """SwitchMix Eq. 5: trao đổi biên độ tần số thấp bằng FFT với 1 ảnh IR thật (unpaired)."""
    if target_ir_t.shape[-2:] != source_t.shape[-2:]:
        target_ir_t = F.interpolate(
            target_ir_t.unsqueeze(0), size=source_t.shape[-2:], mode="bilinear", align_corners=False
        ).squeeze(0)
    fft_src = torch.fft.fft2(source_t, dim=(-2, -1))
    fft_tgt = torch.fft.fft2(target_ir_t, dim=(-2, -1))
    amp_src = torch.fft.fftshift(torch.abs(fft_src), dim=(-2, -1))
    amp_tgt = torch.fft.fftshift(torch.abs(fft_tgt), dim=(-2, -1))
    phase_src = torch.angle(fft_src)

    _, h, w = source_t.shape
    bh = max(1, int(math.floor(h * beta)))
    bw = max(1, int(math.floor(w * beta)))
    ch, cw = h // 2, w // 2
    mask = torch.zeros((1, h, w), device=source_t.device, dtype=source_t.dtype)
    mask[:, max(0, ch - bh):min(h, ch + bh), max(0, cw - bw):min(w, cw + bw)] = phi

    mixed_amp = torch.fft.ifftshift(mask * amp_tgt + (1.0 - mask) * amp_src, dim=(-2, -1))
    reconstructed = torch.fft.ifft2(mixed_amp * torch.exp(1j * phase_src), dim=(-2, -1)).real
    endpoint = reconstructed.clamp(0.0, 1.0)
    return (alpha * source_t + (1.0 - alpha) * endpoint).clamp(0.0, 1.0)


In [ ]:
# Cell 3: Visualize 5 cột trên 1 hàng: RGB | IR thật | IR_GAN gốc | Hướng 2 (GAN-Low+RM-High) | SwitchMix FDA
import matplotlib.pyplot as plt
import matplotlib.patches as patches

device = torch.device(CONFIG["device"] if (CONFIG["device"] == "cuda" and torch.cuda.is_available()) else "cpu")
print("Running visualization on device:", device)

torch.manual_seed(CONFIG["seed"])
samples = load_flir_samples(CONFIG)
gan_translator = load_gan_translator(CONFIG["gan_weights"], device)

n_rows = len(samples)
fig, axes = plt.subplots(n_rows, 5, figsize=(25, 4.3 * n_rows), squeeze=False)
computed = []

for row, sample in enumerate(samples):
    rgb_t = sample["rgb_t"].to(device)
    ir_t = sample["ir_t"].to(device)
    # Dùng 1 ảnh IR khác làm donor unpaired cho FDA (tránh rò rỉ cặp ảnh cùng cảnh)
    donor_ir_t = samples[(row + 1) % n_rows]["ir_t"].to(device)

    gan_t = gan_translator(rgb_t.unsqueeze(0)).squeeze(0)
    mono_t, weights = make_randommono(rgb_t)
    fusion_t = apply_gan_low_randommono_high(
        rgb_t, gan_t, mono_t,
        alpha=CONFIG["alpha"],
        sigma=CONFIG["fusion_sigma"],
        high_weight=CONFIG["fusion_high_weight"],
    )
    fda_t = apply_switchmix_fda(
        mono_t, donor_ir_t,
        beta=CONFIG["fda_beta"],
        phi=CONFIG["fda_phi"],
        alpha=CONFIG["alpha"],
    )
    computed.append((sample, rgb_t.cpu(), ir_t.cpu(), gan_t.cpu(), mono_t.cpu(), fusion_t.cpu(), weights))

    def to_img(t):
        return t.detach().cpu().permute(1, 2, 0).numpy().clip(0.0, 1.0)

    axes[row, 0].imshow(to_img(rgb_t))
    if CONFIG["show_gt_boxes"]:
        for obj in sample["objects"]:
            x1, y1, x2, y2 = obj["box"]
            color = "lime" if obj["label"] == "person" else ("cyan" if obj["label"] == "bicycle" else "yellow")
            axes[row, 0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.2, edgecolor=color, facecolor="none"))
    labels_str = ", ".join(sorted({o["label"] for o in sample["objects"]}))
    axes[row, 0].set_title(f"1. RGB Original\n{sample['stem']} ({labels_str})", fontsize=9)

    axes[row, 1].imshow(to_img(ir_t)[:, :, 0], cmap="gray", vmin=0.0, vmax=1.0)
    axes[row, 1].set_title("2. IR Aligned (Real Thermal)", fontsize=9)

    axes[row, 2].imshow(to_img(gan_t)[:, :, 0], cmap="gray", vmin=0.0, vmax=1.0)
    axes[row, 2].set_title("3. IR_GAN (CycleGAN Raw)", fontsize=9)

    axes[row, 3].imshow(to_img(fusion_t))
    axes[row, 3].set_title(
        f"4. GAN-Low + RM-High (Hướng 2)\nalpha={CONFIG['alpha']}, sigma={CONFIG['fusion_sigma']}, lam={CONFIG['fusion_high_weight']}, w={weights}",
        fontsize=9,
    )

    axes[row, 4].imshow(to_img(fda_t))
    axes[row, 4].set_title(
        f"5. RandomMono + SwitchMix FDA\nalpha={CONFIG['alpha']}, beta={CONFIG['fda_beta']}, phi={CONFIG['fda_phi']}",
        fontsize=9,
    )

    for col in range(5):
        axes[row, col].axis("off")

plt.tight_layout()
if CONFIG["save_path"]:
    out_path = Path(CONFIG["save_path"])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print("Saved 5-column comparison ->", out_path)
plt.show()


In [ ]:
# Cell 4: Zoom sát vào vật thể nhỏ (person / bicycle) & so sánh các mức sigma (1.5, 3.0, 5.0)
SIGMAS = (1.5, 3.0, 5.0)

# Chọn mẫu đầu tiên có vật thể nhỏ nhất thuộc lớp person hoặc bicycle
chosen = None
for item in computed:
    sample = item[0]
    small_objs = [o for o in sample["objects"] if o["label"] in {"person", "bicycle"}]
    if small_objs:
        smallest = min(small_objs, key=lambda o: o["area"])
        chosen = (item, smallest)
        break

if chosen is None:
    print("Không có vật thể person/bicycle trong batch hiện tại để zoom.")
else:
    (sample, rgb_t, ir_t, gan_t, mono_t, _, weights), obj = chosen
    _, H, W = rgb_t.shape
    x1, y1, x2, y2 = obj["box"]
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    half = max(48.0, max(x2 - x1, y2 - y1) * 1.8)
    cx1, cy1 = max(0, int(cx - half)), max(0, int(cy - half))
    cx2, cy2 = min(W, int(cx + half)), min(H, int(cy + half))

    def crop_np(t):
        return t[:, cy1:cy2, cx1:cx2].permute(1, 2, 0).numpy().clip(0.0, 1.0)

    cols = 3 + len(SIGMAS)
    fig, axes = plt.subplots(1, cols, figsize=(4.2 * cols, 4.4), squeeze=False)
    panels = [
        (crop_np(rgb_t), f"RGB Zoom ({obj['label']})\nbox={int(x2-x1)}x{int(y2-y1)}px", False),
        (crop_np(ir_t)[:, :, 0], "IR Real Zoom", True),
        (crop_np(gan_t)[:, :, 0], "IR_GAN Raw Zoom", True),
    ]
    for s in SIGMAS:
        fused = apply_gan_low_randommono_high(
            rgb_t, gan_t, mono_t,
            alpha=CONFIG["alpha"], sigma=s, high_weight=CONFIG["fusion_high_weight"],
        )
        panels.append((crop_np(fused), f"Hướng 2 (sigma={s})\nlam={CONFIG['fusion_high_weight']}", False))

    for idx, (img, title, is_gray) in enumerate(panels):
        if is_gray:
            axes[0, idx].imshow(img, cmap="gray", vmin=0.0, vmax=1.0)
        else:
            axes[0, idx].imshow(img)
        axes[0, idx].set_title(title, fontsize=10)
        axes[0, idx].axis("off")

    plt.tight_layout()
    if CONFIG["zoom_save_path"]:
        zpath = Path(CONFIG["zoom_save_path"])
        zpath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(zpath, dpi=150, bbox_inches="tight")
        print("Saved small-object zoom comparison ->", zpath)
    plt.show()
